# Shadow Testing: Multi-Model Migration Evaluation

Shadow testing lets you evaluate candidate models using **real production traffic** before committing to a migration. Instead of synthetic benchmarks, you compare models on the actual queries your system handles every day.

This notebook walks through a realistic shadow testing workflow:

1. **Simulate production traffic** — 50 healthcare Q&A invocations through the current production model
2. **Capture invocation logs** — Record every request and response as structured logs
3. **Sample from logs** — Randomly select 10 representative samples for evaluation
4. **Evaluate target models** — Run sampled prompts against 4 candidate models
5. **Rank on a leaderboard** — Compare accuracy (LLM-as-a-Judge), latency, and cost

## Architecture

![Shadow Testing Architecture](data/shadow-test-architecture.png)

---

## Step 1: Setup

We import helper functions from `src/utils.py` for model invocation, LLM-as-a-Judge evaluation, safety checks, and model comparison. These are the same functions used in Notebook 2 (Model Evaluation).

In [1]:
import json
import time
import random
from typing import Dict, List
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import sys
sys.path.insert(0, '..')

from src.utils import (
    create_bedrock_client,
    invoke_model_for_evaluation,
    run_model_evaluation,
    compare_models,
    EvaluationResult,
    QUALITY_METRICS,
    print_invocation_log_summary,
    print_evaluation_progress,
    print_evaluation_summary,
    print_migration_recommendation,
)

REGION = "us-east-1"
bedrock = create_bedrock_client(REGION)

# Production model — currently serving healthcare Q&A traffic
PRODUCTION_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
PRODUCTION_MODEL_NAME = "Amazon Nova 2 Lite"

# Target models to evaluate as potential replacements
TARGET_MODELS = {
    "qwen.qwen3-coder-30b-a3b-v1:0": "Qwen 3 Coder",
    "us.amazon.nova-2-lite-v1:0": "Amazon Nova 2 Lite",
    "openai.gpt-oss-20b-1:0": "OpenAI GPT-OSS 20B",
    "us.anthropic.claude-haiku-4-5-20251001-v1:0": "Claude Haiku 4.5",
}

# Judge models for LLM-as-a-Judge evaluation
JUDGE_MODEL_IDS = [
    "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "us.amazon.nova-pro-v1:0",
    "moonshotai.kimi-k2.5",
]

# How many samples to evaluate
SAMPLE_SIZE = 10

print(f"Production model:  {PRODUCTION_MODEL_NAME}")
print(f"Target models:     {len(TARGET_MODELS)}")
for mid, name in TARGET_MODELS.items():
    print(f"  - {name}")
print(f"Judge models:      {len(JUDGE_MODEL_IDS)}")
print(f"Sample size:       {SAMPLE_SIZE}")

Production model:  Amazon Nova 2 Lite
Target models:     4
  - Qwen 3 Coder
  - Amazon Nova 2 Lite
  - OpenAI GPT-OSS 20B
  - Claude Haiku 4.5
Judge models:      3
Sample size:       10


---

## Step 2: Simulate Production Traffic (50 Invocations)

We load 50 healthcare Q&A prompts that represent realistic production traffic — covering medication information, symptom recognition, disease education, and emergency scenarios.

Each prompt is sent to the **production model** (Nova 2 Lite). The request, response, latency, token counts, and cost are captured as an **invocation log entry** — mirroring what Amazon Bedrock's invocation logging would capture in a real deployment.

In [2]:
# Load the 50 healthcare prompts
with open("data/shadow_traffic_dataset.json") as f:
    traffic_dataset = json.load(f)

print(f"Loaded {len(traffic_dataset)} healthcare prompts")
print(f"\nCategories:")
categories = {}
for p in traffic_dataset:
    categories[p['category']] = categories.get(p['category'], 0) + 1
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

Loaded 50 healthcare prompts

Categories:
  disease_education: 17
  emergency_recognition: 10
  medication_info: 16
  symptom_recognition: 7


In [3]:
@dataclass
class InvocationLog:
    """A single invocation log entry from production traffic."""
    request_id: str
    timestamp: float
    model_id: str
    prompt: str
    response: str
    category: str
    golden_answer: str
    critical_entities: List[str]
    input_tokens: int
    output_tokens: int
    latency_ms: float
    cost: float


# Run all 50 prompts through the production model and capture invocation logs
invocation_logs: List[InvocationLog] = []

for i, p in enumerate(traffic_dataset, 1):
    response_text, input_tokens, output_tokens, latency_ms, cost = invoke_model_for_evaluation(
        prompt=p["prompt"],
        model_id=PRODUCTION_MODEL_ID,
        bedrock_client=bedrock,
        region=REGION,
    )

    log = InvocationLog(
        request_id=p["id"],
        timestamp=time.time(),
        model_id=PRODUCTION_MODEL_ID,
        prompt=p["prompt"],
        response=response_text,
        category=p["category"],
        golden_answer=p["golden_answer"],
        critical_entities=p["critical_entities"],
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        latency_ms=latency_ms,
        cost=cost,
    )
    invocation_logs.append(log)

    status = "ERR" if response_text.startswith("Error:") else "OK"
    print(f"  [{i:2d}/50] {p['id']} {status}  {latency_ms:6.0f}ms  {output_tokens:4d} tok  ${cost:.6f}  {p['category']}")

print(f"\nCaptured {len(invocation_logs)} invocation log entries")

  [50/50] ST-050 OK    3312ms   500 tok  $0.000125  emergency_recognition

Captured 50 invocation log entries


In [5]:
print_invocation_log_summary(invocation_logs, categories)

INVOCATION LOG SUMMARY
  Total invocations:   50
  Errors:              0
  Total output tokens: 24,570
  Avg latency:         3417 ms
  Total cost:          $0.006152
  Avg cost/query:      $0.000123

  Category breakdown:
    disease_education          17 logs  avg 3453ms
    emergency_recognition      10 logs  avg 3439ms
    medication_info            16 logs  avg 3421ms
    symptom_recognition         7 logs  avg 3291ms


---

## Step 3: Sample from Invocation Logs

In practice, you wouldn't evaluate every single production request — that would be expensive (each sample requires 4 target model calls + 3 judge calls per target). Instead, you take a **random sample** that represents the traffic mix.

We randomly select 10 invocation log entries. This is the subset we'll evaluate against all target models.

In [6]:
# Randomly sample from invocation logs
random.seed(42)  # Reproducible sampling
sampled_logs = random.sample(invocation_logs, SAMPLE_SIZE)

print(f"Sampled {len(sampled_logs)} entries from {len(invocation_logs)} invocation logs\n")

sample_rows = []
for log in sampled_logs:
    sample_rows.append({
        "ID": log.request_id,
        "Category": log.category,
        "Prompt": log.prompt[:70] + "...",
        "Prod Latency": f"{log.latency_ms:.0f}ms",
        "Prod Tokens": log.output_tokens,
        "Prod Cost": f"${log.cost:.6f}",
    })

df_samples = pd.DataFrame(sample_rows)
print(df_samples.to_string(index=False))

Sampled 10 entries from 50 invocation logs

    ID              Category                                                                    Prompt Prod Latency  Prod Tokens Prod Cost
ST-041       medication_info What are the risks and benefits of hormone replacement therapy (HRT) f...       3670ms          500 $0.000125
ST-008       medication_info Explain how statins work to lower cholesterol and list common side eff...       2829ms          408 $0.000103
ST-002   symptom_recognition Explain the symptoms of type 2 diabetes that patients should watch for...       3500ms          500 $0.000125
ST-018   symptom_recognition A patient reports persistent lower back pain for 3 weeks. What questio...       3271ms          500 $0.000126
ST-016     disease_education Explain what a colonoscopy is, how to prepare, and what to expect duri...       3884ms          500 $0.000125
ST-015   symptom_recognition What are the symptoms of pneumonia and how does it differ from a commo...       3801ms       

---

## Step 4: Evaluate Target Models

For each of the 10 sampled prompts, we invoke all 4 target models and evaluate their responses using:

- **LLM-as-a-Judge** — 3 judge models score each response on 6 quality dimensions (correctness, completeness, relevance, format, coherence, following instructions). Majority vote determines PASS/FAIL.
- **Medical safety check** — Verifies that critical medical entities (drug names, symptoms, warnings) are present in the response.
- **Latency and cost** — Captured directly from each invocation.

This uses `run_model_evaluation()` from `src/utils.py`, the same pipeline from Notebook 2.

In [7]:
# Evaluate all target models on the sampled prompts (parallelized per model)
import concurrent.futures

evaluation_results: Dict[str, List[EvaluationResult]] = {}

for model_id, model_name in TARGET_MODELS.items():
    print(f"\nEvaluating: {model_name}")

    # Run all sampled prompts in parallel for this model
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(sampled_logs)) as executor:
        future_to_log = {
            executor.submit(
                run_model_evaluation,
                prompt_id=log.request_id,
                user_prompt=log.prompt,
                model_id=model_id,
                judge_model_ids=JUDGE_MODEL_IDS,
                golden_answer=log.golden_answer,
                critical_entities=log.critical_entities,
                bedrock_client=bedrock,
                region=REGION,
            ): log
            for log in sampled_logs
        }

        model_results = []
        for future in concurrent.futures.as_completed(future_to_log):
            log = future_to_log[future]
            result = future.result()
            model_results.append((log, result))
            print_evaluation_progress(result, log.request_id, log.category)

    # Preserve original sample order
    log_order = {log.request_id: i for i, log in enumerate(sampled_logs)}
    model_results.sort(key=lambda x: log_order[x[0].request_id])
    evaluation_results[model_id] = [r for _, r in model_results]

print_evaluation_summary(evaluation_results, TARGET_MODELS, len(sampled_logs))

  ST-016 (disease_education)... PASS  quality=4.94  safety=0.60  4389ms  $0.001596

Evaluation complete: 4 models x 10 samples
  Qwen 3 Coder: 10/10 PASS
  Amazon Nova 2 Lite: 10/10 PASS
  OpenAI GPT-OSS 20B: 9/10 PASS
  Claude Haiku 4.5: 10/10 PASS


---

## Step 5: Leaderboard

We aggregate the evaluation results into a **leaderboard** that ranks each target model on three axes:

| Axis | What it measures | How |
|------|-----------------|-----|
| **Accuracy** | Response quality and medical safety | LLM-as-a-Judge pass rate + avg quality score + safety score |
| **Latency** | Response speed | Average time-to-last-byte (ms) |
| **Cost** | Per-query expense | Average cost per invocation ($) |

In [9]:
# Build the leaderboard
comparison = compare_models(evaluation_results)

# Production baseline for reference
prod_avg_latency = np.mean([l.latency_ms for l in sampled_logs])
prod_avg_cost = np.mean([l.cost for l in sampled_logs])

leaderboard = []
for model_id, model_name in TARGET_MODELS.items():
    results = evaluation_results[model_id]

    # Accuracy metrics
    pass_rate = comparison.llm_eval_rates.get(model_id, 0)
    quality_scores = comparison.quality_scores.get(model_id, {})
    avg_quality = np.mean(list(quality_scores.values())) if quality_scores else 0
    avg_safety = np.mean([r.safety_score for r in results])

    # Operational metrics
    avg_latency = np.mean([r.ttlb_ms for r in results])
    avg_cost = np.mean([r.response_cost for r in results])
    latency_ratio = avg_latency / prod_avg_latency if prod_avg_latency > 0 else 1.0
    cost_ratio = avg_cost / prod_avg_cost if prod_avg_cost > 0 else 1.0

    # Weighted composite score (balances accuracy vs operational cost)
    # 50% accuracy (pass rate + quality), 25% cost efficiency, 25% latency efficiency
    accuracy_score = (pass_rate * 0.6 + (avg_quality / 5) * 0.4)  # normalized 0-1
    cost_score = 1 / max(cost_ratio, 0.01)     # inverted: lower cost = higher score
    latency_score = 1 / max(latency_ratio, 0.01)  # inverted: lower latency = higher score
    composite = accuracy_score * 0.50 + cost_score * 0.25 + latency_score * 0.25

    leaderboard.append({
        "model_id": model_id,
        "model_name": model_name,
        "pass_rate": pass_rate,
        "avg_quality": avg_quality,
        "avg_safety": avg_safety,
        "avg_latency": avg_latency,
        "avg_cost": avg_cost,
        "latency_ratio": latency_ratio,
        "cost_ratio": cost_ratio,
        "composite": composite,
    })

# Sort by composite score (best overall balance of accuracy + cost + latency)
leaderboard.sort(key=lambda x: x["composite"], reverse=True)

# --- Color-coded leaderboard table using plotly ---

def metric_color(val, go_thresh, warn_thresh, higher_is_better=True):
    """Return cell color based on thresholds."""
    if higher_is_better:
        if val >= go_thresh: return "#c6efce"
        elif val >= warn_thresh: return "#ffeb9c"
        else: return "#ffc7ce"
    else:
        if val <= go_thresh: return "#c6efce"
        elif val <= warn_thresh: return "#ffeb9c"
        else: return "#ffc7ce"

# Prepare column data
models = [e["model_name"] for e in leaderboard]
composites = [f"{e['composite']:.3f}" for e in leaderboard]
pass_rates = [f"{e['pass_rate']*100:.0f}%" for e in leaderboard]
qualities = [f"{e['avg_quality']:.2f}" for e in leaderboard]
safeties = [f"{e['avg_safety']:.2f}" for e in leaderboard]
latencies = [f"{e['avg_latency']:.0f}" for e in leaderboard]
lat_ratios = [f"{e['latency_ratio']:.2f}x" for e in leaderboard]
costs = [f"${e['avg_cost']:.6f}" for e in leaderboard]
cost_ratios = [f"{e['cost_ratio']:.2f}x" for e in leaderboard]
monthlies = [f"${e['avg_cost'] * 100_000:,.2f}" for e in leaderboard]

# Cell colors per column
white = ["#ffffff"] * len(leaderboard)
pass_colors = [metric_color(e["pass_rate"]*100, 80, 60, True) for e in leaderboard]
quality_colors = [metric_color(e["avg_quality"], 3.5, 2.5, True) for e in leaderboard]
safety_colors = [metric_color(e["avg_safety"], 0.80, 0.60, True) for e in leaderboard]
lat_ratio_colors = [metric_color(e["latency_ratio"], 1.5, 2.5, False) for e in leaderboard]
cost_ratio_colors = [metric_color(e["cost_ratio"], 1.5, 3.0, False) for e in leaderboard]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Model</b>", "<b>Score</b>", "<b>Pass Rate</b>", "<b>Quality</b>", "<b>Safety</b>",
                "<b>Latency (ms)</b>", "<b>Latency Ratio</b>",
                "<b>Cost/Query</b>", "<b>Cost Ratio</b>", "<b>Monthly 100K</b>"],
        fill_color="#4a4a4a",
        font=dict(color="white", size=12),
        align="center",
        height=32,
    ),
    cells=dict(
        values=[models, composites, pass_rates, qualities, safeties,
                latencies, lat_ratios, costs, cost_ratios, monthlies],
        fill_color=[white, white, pass_colors, quality_colors, safety_colors,
                    white, lat_ratio_colors, white, cost_ratio_colors, white],
        font=dict(color="black", size=12),
        align="center",
        height=28,
    ),
)])

fig.update_layout(
    title="Shadow Test Leaderboard (ranked by composite score: 50% accuracy, 25% cost, 25% latency)",
    height=50 + 32 + 28 * len(leaderboard) + 40,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig.show()

In [10]:
# Quality breakdown by metric — plotly table
metric_labels = [m.replace('_', ' ').title() for m in QUALITY_METRICS]

models_col = [e["model_name"] for e in leaderboard]
pass_col = [f"{e['pass_rate']*100:.0f}%" for e in leaderboard]
safety_col = [f"{e['avg_safety']:.2f}" for e in leaderboard]
latency_col = [f"{e['avg_latency']:.0f}" for e in leaderboard]
lat_ratio_col = [f"{e['latency_ratio']:.2f}x" for e in leaderboard]

metric_cols = []
metric_colors = []
for metric in QUALITY_METRICS:
    vals = []
    colors = []
    for entry in leaderboard:
        scores = comparison.quality_scores.get(entry["model_id"], {})
        v = scores.get(metric, 0)
        vals.append(f"{v:.2f}")
        colors.append(metric_color(v, 3.5, 2.5, True))
    metric_cols.append(vals)
    metric_colors.append(colors)

fig2 = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Model</b>", "<b>Pass Rate</b>"] + [f"<b>{m}</b>" for m in metric_labels] + ["<b>Safety</b>", "<b>Latency (ms)</b>", "<b>Latency Ratio</b>"],
        fill_color="#4a4a4a",
        font=dict(color="white", size=12),
        align="center",
        height=32,
    ),
    cells=dict(
        values=[models_col, pass_col] + metric_cols + [safety_col, latency_col, lat_ratio_col],
        fill_color=[
            ["white"] * len(leaderboard),
            [metric_color(e["pass_rate"]*100, 80, 60, True) for e in leaderboard],
        ] + metric_colors + [
            [metric_color(e["avg_safety"], 0.80, 0.60, True) for e in leaderboard],
            ["white"] * len(leaderboard),
            [metric_color(e["latency_ratio"], 1.5, 2.5, False) for e in leaderboard],
        ],
        font=dict(color="black", size=12),
        align="center",
        height=28,
    ),
)])

fig2.update_layout(
    title="Quality Breakdown by Metric",
    height=50 + 32 + 28 * len(leaderboard) + 40,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig2.show()

---

## Step 6: Migration Recommendation

Based on the leaderboard, we identify the best candidate for migration — the model with the strongest combination of accuracy, safety, cost, and latency for this healthcare Q&A workload.

In [11]:
print_migration_recommendation(
    best=leaderboard[0],
    prod_model_name=PRODUCTION_MODEL_NAME,
    prod_avg_latency=prod_avg_latency,
    prod_avg_cost=prod_avg_cost,
    quality_rec_model_id=comparison.recommended_model,
    model_names=TARGET_MODELS,
)

  MIGRATION RECOMMENDATION

  Recommended model:   Qwen 3 Coder
  Composite score:     1.182

  Accuracy
    Jury pass rate:    100%
    Avg quality:       4.91/5
    Safety score:      0.88

  Latency
    Avg response time: 2150ms  (38% faster than production)
    Production:        3457ms

  Cost
    Per query:         $0.000106  (12% cheaper than production)
    Production:        $0.000121
    Monthly (100K):    $10.64  (-$1.43/month)
    Production:        $12.07


  Note: Quality-only ranking recommends Claude Haiku 4.5.
  Composite ranking (accuracy + cost + latency) recommends Qwen 3 Coder.


---

## Traffic Migration Strategy

Once you've identified your target model, migrate production traffic gradually using canary deployment:

| Phase | Traffic to Target | Duration | Success Criteria |
|-------|-------------------|----------|------------------|
| Canary | 5% | 1-2 days | No increase in errors, latency within bounds |
| Early Adopters | 25% | 3-5 days | Quality maintained, user feedback positive |
| Majority | 75% | 1 week | All metrics stable, cost projections confirmed |
| Full Migration | 100% | Ongoing | Source model decommissioned |

**Rollback triggers** — automatically revert if:
- Error rate increases by more than 50%
- P95 latency exceeds 2x baseline for 5+ minutes
- Safety score drops below 0.60 (critical medical entities missing)

In production, implement the shadow router using feature flags (AWS AppConfig), weighted target groups (ALB), or a custom routing layer in your API gateway.

---

## Summary

This notebook demonstrated a complete shadow testing workflow:

1. **Simulated 50 production invocations** through the current model and captured structured invocation logs
2. **Sampled 10 representative queries** from the logs for cost-effective evaluation
3. **Evaluated 4 target models** using LLM-as-a-Judge (3 judges, 6 quality metrics) and medical safety checks
4. **Ranked models on a leaderboard** by accuracy, latency, and cost

The key insight: **use real traffic, not synthetic benchmarks**. Production queries surface edge cases and distribution patterns that hand-crafted test sets miss. Shadow testing lets you validate a migration decision with data before it impacts users.

| Notebook | What You Learned |
|----------|------------------|
| 1. Drift Detection | How model outputs change over time and across prompt variations |
| 2. Model Evaluation | How to systematically score models using LLM-as-a-Jury |
| 3. Prompt Optimization | How to write prompts that work consistently across models |
| **4. Shadow Testing** | **How to use production traffic to validate a migration decision** |